# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-khaled123/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Before coding anything, I check the two signals my rule idea would lean on -- exactly like the
session did live. Both are checked with a bucket table (with n) and get a one-word verdict:
CONFIRMED, OPPOSITE, MIXED, or FALSE.

In [1]:
import pandas as pd, numpy as np, os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(f"{len(df):,} pages loaded")


30,000 pages loaded


### Signal check 1 -- staleness (behind the refresh flags)

Real flag: `stale_visible_page` fires at `days_since_last_update >= 180`. If staleness really
drives decline, older-since-update pages should decline more often.

In [2]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 89, 179, 364, 100000],
    labels=["0-89 (fresh)", "90-179", "180-364 (stale)", "365+ (very stale)"],
)
sig1 = df.groupby("staleness_bucket", observed=True)["is_declining"].agg(["mean", "count"])
sig1.columns = ["decline_rate", "n"]
print(sig1)
print("\nVerdict: MIXED. Decline rate rises from fresh (0.51) to 90-179 days (0.61) -- so far so good --")
print("but then DROPS at 180-364 days (0.47), and the 365+ bucket has only n=5 (too small to trust).")
print("Staleness alone does not cleanly predict decline in this data -- I am NOT using it in the rule below.")
print("A clearly negative signal check is still useful: it just saved the rule from leaning on noise.")


                   decline_rate      n
staleness_bucket                      
0-89 (fresh)           0.512031  20655
90-179                 0.611057   9171
180-364 (stale)        0.467456    169
365+ (very stale)      0.600000      5

Verdict: MIXED. Decline rate rises from fresh (0.51) to 90-179 days (0.61) -- so far so good --
but then DROPS at 180-364 days (0.47), and the 365+ bucket has only n=5 (too small to trust).
Staleness alone does not cleanly predict decline in this data -- I am NOT using it in the rule below.
A clearly negative signal check is still useful: it just saved the rule from leaning on noise.


### Signal check 2 -- CTR vs. position tier (behind the CTR-fix logic)

Real flag logic: `low_ctr_visible_page` / `ctr_review_candidate` compare a page's CTR to what is
normal for its position tier. If that premise holds, mean CTR should fall as position tier
worsens.

In [3]:
vis = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()
sig2 = vis.groupby("position_tier")["ctr"].agg(["mean", "count"]).sort_values("mean", ascending=False)
sig2.columns = ["mean_ctr", "n"]
print(sig2)
print("\nVerdict: CONFIRMED. Mean CTR falls in a clean staircase from page_1 (0.355) down to")
print("deep (0.055) -- CTR really does depend heavily on position tier, which is exactly the premise")
print("the CTR-fix flag leans on. This signal earns a place in the rule below; staleness did not.")


               mean_ctr     n
position_tier                
page_1         0.354760  8633
top_3          0.334128   533
striking       0.255782  5903
page_3_5       0.142359  6058
deep           0.055415   879

Verdict: CONFIRMED. Mean CTR falls in a clean staircase from page_1 (0.355) down to
deep (0.055) -- CTR really does depend heavily on position tier, which is exactly the premise
the CTR-fix flag leans on. This signal earns a place in the rule below; staleness did not.


### The rule, in plain words

A page is worth a title/meta/snippet review if it already has real visibility (>=100 impressions,
a valid tracked position) but its CTR sits clearly below what pages at its own position tier
normally get. Bigger audience and bigger gap both push it higher up the queue. Staleness is
deliberately left out of the score -- Signal check 1 showed it is not a reliable driver here.

**Reason code (one, constant):** `ctr_below_position_tier_expectation`
**Action (one, constant):** `review_title_meta_snippet`
**Score:** `max(0, expected_ctr_for_tier - actual_ctr) * impressions_90d`

## 2. Build the ranked queue (writes the CSV)

In [4]:
expected_ctr = vis.groupby("position_tier")["ctr"].mean()
vis["expected_ctr_tier"] = vis["position_tier"].map(expected_ctr)
vis["ctr_gap"] = (vis["expected_ctr_tier"] - vis["ctr"]).clip(lower=0)
vis["score"] = vis["ctr_gap"] * vis["impressions_90d"]
vis["reason_code"] = "ctr_below_position_tier_expectation"
vis["action"] = "review_title_meta_snippet"

ranked = vis.sort_values("score", ascending=False).reset_index(drop=True)
print(f"{len(ranked):,} of {len(df):,} pages scored (visible + valid position); "
      f"{(ranked['score'] > 0).sum():,} have a positive CTR gap")

out_cols = ["content_id", "client_id", "content_type", "position_tier", "avg_position", "ctr",
            "expected_ctr_tier", "ctr_gap", "impressions_90d", "score", "reason_code", "action"]
os.makedirs("work/outputs", exist_ok=True)
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv")

ranked[out_cols].head(10)


22,006 of 30,000 pages scored (visible + valid position); 14,797 have a positive CTR gap
Wrote work/outputs/baseline_action_score.csv


,content_id,client_id,content_type,position_tier,avg_position,ctr,expected_ctr_tier,ctr_gap,impressions_90d,score,reason_code,action
0,content_5fe46e04994d,client_4e07408562,keyword article,page_1,4.2,0.14,0.354760,0.214760,517715,111184.288695,ctr_below_position_tier_expectation,review_title_meta_snippet
1,content_8c19996aa890,client_4e07408562,keyword article,top_3,2.5,0.15,0.334128,0.184128,509252,93767.338236,ctr_below_position_tier_expectation,review_title_meta_snippet
2,content_36ff89c8214e,client_19581e27de,keyword article,page_1,7.3,0.05,0.354760,0.304760,295097,89933.656438,ctr_below_position_tier_expectation,review_title_meta_snippet
3,content_8451fc6f034d,client_d029fa3a95,keyword article,top_3,2.3,0.03,0.334128,0.304128,272144,82766.496060,ctr_below_position_tier_expectation,review_title_meta_snippet
4,content_c8e9d6ab9013,client_19581e27de,keyword article,page_1,9.7,0.00,0.354760,0.354760,208678,74030.532830,ctr_below_position_tier_expectation,review_title_meta_snippet
5,content_c84a0ab98e90,client_f369cb89fc,keyword article,page_1,7.8,0.03,0.354760,0.324760,223271,72509.410303,ctr_below_position_tier_expectation,review_title_meta_snippet
6,content_cb112fce36be,client_19581e27de,keyword article,page_1,5.6,0.16,0.354760,0.194760,309910,60357.961033,ctr_below_position_tier_expectation,review_title_meta_snippet
7,content_73c54f78c06a,client_f369cb89fc,keyword article,page_1,4.7,0.10,0.354760,0.254760,213963,54509.137544,ctr_below_position_tier_expectation,review_title_meta_snippet
8,content_aaef01a50def,client_19581e27de,keyword article,page_1,5.4,0.25,0.354760,0.104760,517109,54172.154351,ctr_below_position_tier_expectation,review_title_meta_snippet
9,content_1a9e894be2e2,client_19581e27de,keyword article,page_1,4.0,0.23,0.354760,0.124760,416180,51922.468319,ctr_below_position_tier_expectation,review_title_meta_snippet


## 3. Top-10 review

One line each: the action, why it is there, and what would make it wrong.

In [5]:
notes = [
    "Page 1 avg (pos 4.2) but CTR 0.14 vs 0.355 tier norm, 517k impressions -> huge, cheap-to-fix audience. Wrong if the meta/snippet was already just refreshed and this is stale cache.",
    "Top-3 avg (pos 2.5), CTR 0.15 vs 0.334, 509k impressions -> same pattern, second-largest audience. Wrong if a competitor's rich snippet (review stars, FAQ) is structurally outranking the click, not the title.",
    "Page 1 avg (pos 7.3), CTR 0.05 vs 0.355 -> very low absolute CTR for page 1. Wrong if intent mismatch (page ranks for a query it does not actually answer) rather than a fixable title/meta issue.",
    "Top-3 avg (pos 2.3), CTR 0.03 -> near-zero clicks despite a strong rank. Wrong if this is a very recent ranking jump and CTR just has not caught up yet (needs more days of data before acting).",
    "Page 1 avg (pos 9.7), CTR 0.00 -> largest raw gap. Wrong (my top weak pick): position 9.7 sits at the bottom edge of the page_1 tier, so comparing it to the TIER AVERAGE (positions ~1-10) overstates what a page at position ~10 should realistically get.",
    "Page 1 avg (pos 7.8), CTR 0.03 vs 0.355 -> same shape as row 3/5. Wrong if this client's SERP snippet is a rich result (e.g. video) that naturally suppresses plain-text CTR.",
    "Page 1 avg (pos 5.6), CTR 0.16 -> moderate gap, still large audience (310k). Wrong if this is a branded query where users already know the destination and click a different, more specific result on purpose.",
    "Page 1 avg (pos 4.7), CTR 0.10 vs 0.355 -> mid-position, mid-gap, real audience. Wrong if seasonal demand dropped so CTR reflects fewer relevant searchers, not a bad title.",
    "Page 1 avg (pos 5.4), CTR 0.25 -> smallest gap in the top 10 (0.10) but the single largest impression volume (517k), so total opportunity is still large. Wrong if 0.25 CTR is actually fine for this specific query's intent (e.g. informational, low commercial urge to click through).",
    "Page 1 avg (pos 4.0), CTR 0.23 -> similar shape to row 9. Wrong if this page already has an A/B test running on its title and this snapshot is mid-test.",
]
top10 = ranked.head(10).reset_index(drop=True)
for i, note in enumerate(notes):
    row = top10.loc[i]
    print(f"{i+1}. {row['content_id']} (client {row['client_id']}) -- action: {row['action']}")
    print(f"   why: pos_tier={row['position_tier']} (avg {row['avg_position']}), ctr={row['ctr']:.2f} vs "
          f"tier norm {row['expected_ctr_tier']:.3f}, impressions={row['impressions_90d']:,}, score={row['score']:.0f}")
    print(f"   what would make it wrong: {note}")
    print()


1. content_5fe46e04994d (client client_4e07408562) -- action: review_title_meta_snippet
   why: pos_tier=page_1 (avg 4.2), ctr=0.14 vs tier norm 0.355, impressions=517,715, score=111184
   what would make it wrong: Page 1 avg (pos 4.2) but CTR 0.14 vs 0.355 tier norm, 517k impressions -> huge, cheap-to-fix audience. Wrong if the meta/snippet was already just refreshed and this is stale cache.

2. content_8c19996aa890 (client client_4e07408562) -- action: review_title_meta_snippet
   why: pos_tier=top_3 (avg 2.5), ctr=0.15 vs tier norm 0.334, impressions=509,252, score=93767
   what would make it wrong: Top-3 avg (pos 2.5), CTR 0.15 vs 0.334, 509k impressions -> same pattern, second-largest audience. Wrong if a competitor's rich snippet (review stars, FAQ) is structurally outranking the click, not the title.

3. content_36ff89c8214e (client client_19581e27de) -- action: review_title_meta_snippet
   why: pos_tier=page_1 (avg 7.3), ctr=0.05 vs tier norm 0.355, impressions=295,097, score=8

## 4. Weak picks + leakage check

In [6]:
print("Weakest picks in the top 10:")
print("- Row 5 (pos 9.7, CTR 0.00): flagged above as the weakest pick -- comparing it to the whole")
print("  page_1 tier average is too coarse near the tier boundary; a finer expected-CTR-by-exact-position")
print("  curve (not 5 buckets) would be a fairer comparison before anyone spends editor time on it.")
print()
client_counts = top10["client_id"].value_counts()
print("Client concentration in the top 10:")
print(client_counts)
print(f"-> {client_counts.max()} of the top 10 rows all come from a single client "
      f"({client_counts.idxmax()}), all 'keyword article' content_type. Worth a manual sanity check that")
print("  this is real client-level under-performance and not a measurement quirk specific to that client.")

print("\nLeakage check:")
used_cols = ["ctr", "avg_position", "position_tier", "impressions_90d", "content_type"]
print(f"Score uses only: {used_cols}")
print("- None of these are trend_direction / trend_pct (the label-carrying columns from the starter data).")
print("- No future-window data used -- ctr/avg_position/impressions_90d are all trailing-90-day, same-window")
print("  observed signals, not a later outcome.")
print("- No FlyRank product decision flags used (none exist in this dataset in the first place).")
print("Clean: this rule is safe to use as the Week-5 model's baseline to beat.")


Weakest picks in the top 10:
- Row 5 (pos 9.7, CTR 0.00): flagged above as the weakest pick -- comparing it to the whole
  page_1 tier average is too coarse near the tier boundary; a finer expected-CTR-by-exact-position
  curve (not 5 buckets) would be a fairer comparison before anyone spends editor time on it.

Client concentration in the top 10:
client_id
client_19581e27de    5
client_4e07408562    2
client_f369cb89fc    2
client_d029fa3a95    1
Name: count, dtype: int64
-> 5 of the top 10 rows all come from a single client (client_19581e27de), all 'keyword article' content_type. Worth a manual sanity check that
  this is real client-level under-performance and not a measurement quirk specific to that client.

Leakage check:
Score uses only: ['ctr', 'avg_position', 'position_tier', 'impressions_90d', 'content_type']
- None of these are trend_direction / trend_pct (the label-carrying columns from the starter data).
- No future-window data used -- ctr/avg_position/impressions_90d are a

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.